# Azure ML Batch Endpoint für die Anti-Cheat-Pipeline

Dieses Notebook richtet den Machine-Learning-Teil der Pipeline ein. Es verbindet sich mit einem Azure ML Workspace, registriert ein Klassifikationsmodell, erstellt Compute und Batch Endpoint, konfiguriert das Deployment und führt einen Testlauf aus.

Vor der Ausführung müssen die Platzhalter für Subscription, Ressourcengruppe und Workspace ersetzt sowie `model.pkl`, `conda.yaml` und eine passende `sample-data.csv` bereitgestellt werden.


# 1. Connect to Azure Machine Learning Workspace

The [workspace](https://docs.microsoft.com/en-us/azure/machine-learning/concept-workspace) is the top-level resource for Azure Machine Learning, providing a centralized place to work with all the artifacts you create when you use Azure Machine Learning. In this section we will connect to the workspace in which the job will be run.

## 1.1. Import the required libraries

In [ ]:
%pip install azure-ai-ml azure-identity --use-deprecated=legacy-resolver

In [ ]:
from azure.ai.ml import MLClient, Input
from azure.ai.ml.entities import (
    BatchEndpoint,
    ModelBatchDeployment,
    ModelBatchDeploymentSettings,
    Model,
    Environment,
    AmlCompute,
    BatchRetrySettings,
    CodeConfiguration,
)
from azure.identity import DefaultAzureCredential
from azure.ai.ml.constants import AssetTypes, BatchDeploymentOutputAction

## 1.2 Workspace konfigurieren

Subscription, Ressourcengruppe und Workspace werden über neutrale Platzhalter konfiguriert. `DefaultAzureCredential` verwendet die lokal beziehungsweise in Azure verfügbare Identität.


In [ ]:
subscription_id = "YOUR_AZURE_SUBSCRIPTION_ID"
resource_group = "YOUR_RESOURCE_GROUP"
workspace = "YOUR_AZURE_ML_WORKSPACE"

In [ ]:
ml_client = MLClient(
    DefaultAzureCredential(), subscription_id, resource_group, workspace
)

# 2. Batch Endpoint erstellen

Der Batch Endpoint nimmt Verweise auf Feature-Dateien entgegen, führt die Inferenz auf dem konfigurierten Compute aus und schreibt das Ergebnis in einen Azure-ML-Datastore.

## 2.1 Endpoint konfigurieren


In [ ]:
endpoint_name = "anti-cheat-batch"

In [ ]:
import random
import string

# Creating a unique endpoint name by including a random suffix
allowed_chars = string.ascii_lowercase + string.digits
endpoint_suffix = "".join(random.choice(allowed_chars) for x in range(5))
endpoint_name = f"{endpoint_name}-{endpoint_suffix}"

print(f"Endpoint name: {endpoint_name}")

Let's configure the endpoint:

In [ ]:
endpoint = BatchEndpoint(
    name=endpoint_name,
    description="Anti Cheat Model",
    tags={"type": "batch"},
)

## 2.2 Create the endpoint
Using the `MLClient` created earlier, we will now create the Endpoint in the workspace. This command will start the endpoint creation and return a confirmation response while the endpoint creation continues.

In [ ]:
ml_client.begin_create_or_update(endpoint).result()

## 3. Modell registrieren

### 3.1 Modellartefakt

Das trainierte Klassifikationsmodell wird als `model.pkl` registriert. Sein Feature-Schema muss mit der in `docs/DATENMODELL.md` dokumentierten Eingabe übereinstimmen.


### 3.2 Registering the model in the workspace

We need to register the model in order to use it with Azure Machine Learning:

In [ ]:
model_name = "anti-cheat-classifier"
model_local_path = "model.pkl"

model = ml_client.models.create_or_update(
    Model(
        name=model_name,
        path=model_local_path,
        type=AssetTypes.CUSTOM_MODEL,
        tags={"task": "classification", "framework": "scikit-learn"},
    )
)

Let's get a reference to the model:

In [ ]:
model = ml_client.models.get(name=model_name, label="latest")

# 4. Create a deployment
A deployment is a set of resources required for hosting the model that does the actual inferencing.

### 4.1 Creating an scoring script to work with the model

In [ ]:
%%writefile batch_driver.py
import os
import pandas as pd
import joblib
import glob
from os.path import basename
from typing import List

def init():
    global model
    print("INIT!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    model_path = os.environ["AZUREML_MODEL_DIR"]

    # Suche rekursiv nach einem .pkl
    pkls = glob.glob(f"{model_path}/**/*.pkl", recursive=True)

    if not pkls:
        raise FileNotFoundError(
            f"No .pkl model found in model directory: {model_path}. "
            f"Files found: {glob.glob(model_path + '/**/*', recursive=True)}"
        )

    model_file = pkls[0]
    print(f"Loading model from: {model_file}")

    model = joblib.load(model_file)


def run(mini_batch: List[str]) -> pd.DataFrame:
    print(f"Executing run method over batch of {len(mini_batch)} files.")
    print("RUN!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    results = []
    
    for file_path in mini_batch:
        
        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            continue

        try:
            preds = model.predict(df)
        except Exception as e:
            print(f"Error during prediction: {e}")
            continue

        row_results = []
        for idx, p in enumerate(preds):
            row_results.append({
                "file": basename(file_path),
                "row": idx,
                "prediction": p
            })

        results.extend(row_results)

    return pd.DataFrame(results)


### 4.2 Creating the compute

Batch deployments can run on any Azure ML compute that already exists in the workspace. That means that multiple batch deployments can share the same compute infrastructure. In this example, we are going to work on an AzureML compute cluster called `cpu-cluster`. Let's verify the compute exists on the workspace or create it otherwise.

In [ ]:
compute_name = "anti-cheat-cpu-cluster"
if not any(filter(lambda m: m.name == compute_name, ml_client.compute.list())):
    compute_cluster = AmlCompute(
        name=compute_name,
        description="CPU cluster compute",
        min_instances=0,
        max_instances=1,
    )
    ml_client.compute.begin_create_or_update(compute_cluster).result()

Compute may take time to be created. Let's wait for it:

In [ ]:
from time import sleep

print(f"Waiting for compute {compute_name}", end="")
while ml_client.compute.get(name=compute_name).provisioning_state == "Creating":
    sleep(1)
    print(".", end="")

print(" [DONE]")

### 4.3 Laufzeitumgebung erstellen

Die Conda-Datei definiert die Python-Bibliotheken des Modells und des Scoring-Skripts. Sie muss gemeinsam mit dem Modell versioniert werden.


In [ ]:
from azure.ai.ml.entities import Environment

env = Environment(
    name="my-env",
    description="Environment for batch scoring",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    conda_file="conda.yaml"  # Pfad zu deiner Conda-Datei
)


### 4.3 Configure the deployment

We will create a deployment for our endpoint using the `BatchDeployment` class. This class allows user to configure the following key aspects.
- `name` - Name of the deployment.
- `endpoint_name` - Name of the endpoint to create the deployment under.
- `model` - The model to use for the deployment. This value can be either a reference to an existing versioned model in the workspace or an inline model specification.
- `environment` - The environment to use for the deployment. This value can be either a reference to an existing versioned environment in the workspace or an inline environment specification.
- `code_path`- Path to the source code directory for scoring the model
- `scoring_script` - Relative path to the scoring file in the source code directory
- `compute` - Name of the compute target to execute the batch scoring jobs on
- `instance_count`- The number of nodes to use for each batch scoring job.		1
- `max_concurrency_per_instance`- The maximum number of parallel scoring_script runs per instance.
- `mini_batch_size`	- The number of files the code_configuration.scoring_script can process in one `run()` call.
- `retry_settings`- Retry settings for scoring each mini batch.		
   - `max_retries`- The maximum number of retries for a failed or timed-out mini batch (default is 3)
   - `timeout`- The timeout in seconds for scoring a mini batch (default is 30)
- `output_action`- Indicates how the output should be organized in the output file. Allowed values are `append_row` or `summary_only`. Default is `append_row`
- `output_file_name`- Name of the batch scoring output file. Default is `predictions.csv`
- `environment_variables`- Dictionary of environment variable name-value pairs to set for each batch scoring job.
- `logging_level`- The log verbosity level.	Allowed values are `warning`, `info`, `debug`. Default is `info`.


In [ ]:
deployment = ModelBatchDeployment(
    name="anti-cheat-deploy",
    description="A deployment using AutoML.",
    endpoint_name=endpoint_name,
    model=model,
    code_configuration=CodeConfiguration(
        code="./", scoring_script="batch_driver.py"
    ),
    environment=env,
    compute=compute_name,
    settings=ModelBatchDeploymentSettings(
        max_concurrency_per_instance=1,
        mini_batch_size=10,
        instance_count=1,
        output_action=BatchDeploymentOutputAction.APPEND_ROW,
        output_file_name="predictions.csv",
        retry_settings=BatchRetrySettings(max_retries=3, timeout=30),
        logging_level="info",
    ),
)

### 4.4 Create the deployment
Using the `MLClient` created earlier, we will now create the deployment in the workspace. This command will start the deployment creation and return a confirmation response while the deployment creation continues.

In [ ]:
ml_client.begin_create_or_update(deployment).result()

Let's update the default deployment name in the endpoint:

In [ ]:
endpoint = ml_client.batch_endpoints.get(endpoint_name)
endpoint.defaults.deployment_name = deployment.name
ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

We can see the details of the deployment as follows:

In [ ]:
ml_client.batch_deployments.get(name=deployment.name, endpoint_name=endpoint.name)

### 4.5 Test the endpoint with sample data
Using the `MLClient` created earlier, we will get a handle to the endpoint. The endpoint can be invoked using the `invoke` command with the following parameters:
- `name` - Name of the endpoint
- `input_path` - Path where input data is present
- `deployment_name` - Name of the specific deployment to test in an endpoint

In [ ]:
import time

# Let's sleep for 2 min to ensure all resources are ready. This is only for automation purposes.
time.sleep(120)

Für den Testlauf wird eine lokale `sample-data.csv` mit demselben Feature-Schema wie das trainierte Modell verwendet.


In [ ]:
from pathlib import Path

dataset_path = Path("./sample-data.csv")
temp_data_path = Path(".")
print(dataset_path)


#### 4.5.1 Invoke the endpoint

Let's now invoke the endpoint for batch scoring job:

In [ ]:
job = ml_client.batch_endpoints.invoke(
    endpoint_name=endpoint_name,
    deployment_name=deployment.name,
    input=Input(
        path=Path(dataset_path),
        type=AssetTypes.URI_FILE,
    ),
)

#### 4.5.2 Get the details of the invoked job
Let us get details and logs of the invoked job

In [ ]:
ml_client.jobs.get(job.name)

We can wait for the job to finish using the following code:

In [ ]:
ml_client.jobs.stream(job.name)

#### 4.5.3 Download the results

The deployment creates a child job that executes the scoring. We can get the details of it using the following code:

In [ ]:
scoring_job = list(ml_client.jobs.list(parent_job_name=job.name))[0]

In [ ]:
print("Job name:", scoring_job.name)
print("Job status:", scoring_job.status)
print(
    "Job duration:",
    scoring_job.creation_context.last_modified_at
    - scoring_job.creation_context.created_at,
)

The outputs generated by the deployment job will be placed in an output named `score`:

In [ ]:
ml_client.jobs.download(name=scoring_job.name, download_path=".", output_name="score")

We can read this data using pandas library:

In [ ]:
import pandas as pd

score = pd.read_csv(
    "named-outputs/score/predictions.csv",
    header=None,
    names=["file", "class"],
    sep=" ",
)
score

### 4.6 Override deployment configuration at invocation time

#### 4.6.1 Output location

You can indicate the output path where you want the job to place the results. To do that, let's first find the ID of a data store registered in AzureML. You can only place outputs in data stores:

In [ ]:
batch_ds = ml_client.datastores.get_default()

Run the job:

In [ ]:
filename = f"predictions-{random.randint(0,99999)}.csv"

job = ml_client.batch_endpoints.invoke(
    endpoint_name=endpoint_name,
    input=Input(
        path=Path(temp_data_path) / "data",
        type=AssetTypes.URI_FOLDER,
    ),
    params_override=[
        {"output_dataset.datastore_id": f"azureml:{batch_ds.id}"},
        {"output_dataset.path": f"/{endpoint_name}/"},
        {"output_file_name": filename},
    ],
)

#### 4.6.1 Override deployment configuration settings

Some other parameters can be override, including `mini_batch_size` and `instance_count`:

In [ ]:
job = ml_client.batch_endpoints.invoke(
    endpoint_name=endpoint_name,
    input=Input(path="./data"),
    params_override=[{"mini_batch_size": "20"}, {"compute.instance_count": "5"}],
)